# 01 — CGM Data Exploration

This notebook explores the OhioT1DM dataset (or the synthetic stand-in) before any modelling.
Goals:
- Understand the data distribution and missing value patterns
- Visualise glucose traces with clinical zone annotations
- Compute Time in Range (TIR) and glycemic variability
- Confirm the windowing strategy makes sense for the prediction horizons

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from data.sample.generate_synthetic import simulate_cgm
from src.preprocessing import CGMProcessor
from src.visualization import plot_glucose_trace

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load data

We use synthetic data here; swap `simulate_cgm()` for `processor.load_ohio_xml()` once you
have OhioT1DM access (request at http://smarthealth.cs.ohio.edu/OhioT1DM-dataset.html).

In [ ]:
df_raw = simulate_cgm(days=14, seed=42)
print(f"Raw: {len(df_raw)} readings  ({df_raw['datetime'].min()} → {df_raw['datetime'].max()})")
df_raw.head()

## 2. Clean and inspect missing values

In [ ]:
processor = CGMProcessor(seq_len=24, pred_horizons=[6, 12])
df = processor.clean(df_raw)

print(f"After cleaning: {len(df)} readings")
print(f"Missing values: {df['glucose'].isna().sum()}")
print(f"\nGlucose summary (mg/dL):")
print(df['glucose'].describe().round(1))

## 3. Distribution and clinical zone breakdown

In [ ]:
tir = processor.time_in_range(df['glucose'].values)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram with clinical zones
ax = axes[0]
ax.hist(df['glucose'], bins=60, color='#60a5fa', edgecolor='white', linewidth=0.3, alpha=0.85)
ax.axvspan(54, 70, alpha=0.25, color='#f97316', label='Hypo L1 (54–70)')
ax.axvspan(0, 54, alpha=0.25, color='#ef4444', label='Hypo L2 (<54)')
ax.axvspan(70, 180, alpha=0.12, color='#4ade80', label='Target (70–180)')
ax.axvspan(180, 250, alpha=0.15, color='#facc15', label='Hyper L1 (180–250)')
ax.axvspan(250, 400, alpha=0.15, color='#ef4444', label='Hyper L2 (>250)')
ax.set_xlabel('Glucose (mg/dL)'); ax.set_ylabel('Count')
ax.set_title('Glucose distribution'); ax.legend(fontsize=8)

# TIR pie
ax2 = axes[1]
labels = ['Target (70–180)', 'Hypo L1', 'Hypo L2', 'Hyper L1', 'Hyper L2']
values = [tir['tir_normal'], tir['tir_hypo_l1'], tir['tir_hypo_l2'],
          tir['tir_hyper_l1'], tir['tir_hyper_l2']]
colors = ['#4ade80', '#f97316', '#ef4444', '#facc15', '#dc2626']
ax2.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Time in Range breakdown')

plt.tight_layout()
plt.savefig('../results/plots/01_tir_distribution.png', bbox_inches='tight')
plt.show()
print("TIR:", {k: f'{v:.1f}%' for k, v in tir.items()})

## 4. Full trace — 3-day window

In [ ]:
three_days = df.head(3 * 288)   # 288 readings per day at 5-min intervals
fig = plot_glucose_trace(
    timestamps=three_days['datetime'],
    glucose=three_days['glucose'].values,
    title='3-Day CGM Trace (synthetic patient)',
    save_path='../results/plots/01_3day_trace.png',
)
plt.show()

## 5. Window shape validation

Confirm that the sliding-window approach produces the correct input/output shapes.

In [ ]:
processor.fit_scaler(df)
X, y = processor.make_windows(df, normalize=True)

print(f"X shape: {X.shape}  → (samples, seq_len=24, features=1)")
print(f"y shape: {y.shape}  → (samples, horizons=2) — [30-min, 60-min]")

# Visualise a random window with its targets
idx = 100
fig, ax = plt.subplots(figsize=(10, 3))
window_mg = processor.inverse_transform(X[idx, :, 0].reshape(1, -1)).flatten()
target_mg = processor.inverse_transform(y[idx:idx+1]).flatten()

ax.plot(range(24), window_mg, 'o-', color='#60a5fa', ms=4, label='Input window (2h)')
ax.plot([24 + 5, 24 + 11], target_mg, 'D', color='#f97316', ms=8,
        label='Targets: 30-min & 60-min')
ax.axhspan(70, 180, alpha=0.07, color='#4ade80')
ax.set_xlabel('Timestep (5-min intervals)'); ax.set_ylabel('Glucose (mg/dL)')
ax.set_title(f'Sample window #{idx} with prediction targets'); ax.legend()
plt.tight_layout()
plt.show()